In [ ]:
import zarr
import s3fs
import numpy as np
import os

from pprint import pprint

# SageMaker
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import TrainingStep, CacheConfig
from sagemaker.pytorch.estimator import PyTorch
from sagemaker.inputs import TrainingInput

In [5]:
%store -r

In [6]:
 # Location constants
ZARR_REF = f"{s3_project_prefix}/store"
TRAIN_REF = "train_indices.npy"
VAL_REF = "val_indices.npy"
INDICES_REF = f"{s3_project_prefix}/indices"
STATE_DICT_CNN = f"{s3_project_prefix}/models/model_artifacts/pipelines-we82lejaox5y-TrainInteriorBoundsM-i8k96yaPTB/output/model.tar.gz"
#STATE_DICT_GAT = f"{s3_project_prefix}/models/model_artifacts/gat/pipelines-w1xso7rkncgk-TrainGATModel-UArmzjkICr/output/model.tar.gz"
#STATE_DICT_GAT = f"{s3_project_prefix}/models/model_artifacts/gat/pipelines-jarvjhd9w2rj-TrainGATModel-aturXwW5fg/output/model.tar.gz"
#STATE_DICT_GAT = f"{s3_project_prefix}/models/model_artifacts/gat/pipelines-g99tojdehnvm-TrainGATModel-oa8lB4rcWR/output/model.tar.gz"
#STATE_DICT_GAT = f"{s3_project_prefix}/models/model_artifacts/gat/pipelines-zdlymkchgpzg-TrainGATModel-iBAHQOcsDX/output/model.tar.gz"
STATE_DICT_GAT = f"{s3_project_prefix}/models/model_artifacts/gat/pipelines-g74eu4gn1fgf-TrainGATModel-ajs5kHDnKX/output/model.tar.gz"

# Data Splitting

In [4]:
# Open the S3 filesystem
fs = s3fs.S3FileSystem()

# Open the zarr store
store = s3fs.S3Map(root=os.path.join(ZARR_REF, "data.zarr"), s3=fs, check=False)
n = zarr.open(store, mode='r')['inside_masks'].shape[0]
indices = np.arange(n)

# Setting seed for reproducibility
np.random.seed(39)
np.random.shuffle(indices)

split = int(n * 0.8)
train_idx = indices[:split]
val_idx   = indices[split:]

# Persist the split across runs
np.save(TRAIN_REF, train_idx)
np.save(VAL_REF,   val_idx)

# Push to S3
fs.put(TRAIN_REF, f"{INDICES_REF}/{TRAIN_REF}")
fs.put(VAL_REF, f"{INDICES_REF}/{VAL_REF}")

ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7f6583106ed0>
ERROR:asyncio:Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7f65831b3170>, 1125.281381403)])']
connector: <aiohttp.connector.TCPConnector object at 0x7f6583106cc0>
ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7f65831068d0>
ERROR:asyncio:Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7f65831b2e70>, 1125.286839393)])']
connector: <aiohttp.connector.TCPConnector object at 0x7f65831068a0>


' np.random.shuffle(indices)\n\nsplit = int(n * 0.8)\ntrain_idx = indices[:split]\nval_idx   = indices[split:]\n\n# Persist the split across runs\nnp.save(TRAIN_REF, train_idx)\nnp.save(VAL_REF,   val_idx)\n\n# Push to S3\nfs.put(TRAIN_REF, f"{INDICES_REF}/{TRAIN_REF}")\nfs.put(VAL_REF, f"{INDICES_REF}/{VAL_REF}") '

# Pipelines

In [6]:
cnn_model_path = f"{s3_project_prefix}/models/model_artifacts/cnn"
gat_model_path = f"{s3_project_prefix}/models/model_artifacts/gat"

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name
pipeline_session = PipelineSession()

# Training Step

In [ ]:
# caching
cache_config = CacheConfig(enable_caching=True, expire_after="PT1H")

pytorch_train_cnn = PyTorch(
    entry_point="train_cnn.py",
    source_dir="./src/",
    role=role,
    framework_version="2.1.0",
    py_version="py310",
    instance_type="ml.g5.4xlarge",
    instance_count=1,
    output_path=cnn_model_path,
    sagemaker_session=pipeline_session,
    environment={"PYTHONUNBUFFERED": "1"},
    hyperparameters={
        "epochs": 12,
        "learning-rate": 1e-4,
        "batch-size": 64,
        "state-dict": STATE_DICT_CNN        
    },
)

train_cnn_args = pytorch_train_cnn.fit(
    inputs={
        "indices": TrainingInput(INDICES_REF, input_mode="File"),
        "zarr": TrainingInput(ZARR_REF, input_mode="FastFile")
    }
)

step_train_cnn = TrainingStep(
    name="TrainInteriorBoundsModel",
    step_args=train_cnn_args,
    cache_config=cache_config
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [ ]:
pytorch_train_gat = PyTorch(
    entry_point="train_gat.py",
    source_dir="./src/",
    role=role,
    framework_version="2.1.0",
    py_version="py310",
    instance_type="ml.g5.4xlarge",
    instance_count=1,
    output_path=gat_model_path,
    sagemaker_session=pipeline_session,
    environment={"PYTHONUNBUFFERED": "1"},
    hyperparameters={
        "epochs": 12,
        "learning-rate": 3e-4,
        "batch-size": 64,
        "dropout": 0.1,
        "state-dict": STATE_DICT_GAT      
    },
)

train_gat_args = pytorch_train_gat.fit(
    inputs={
        "indices": TrainingInput(INDICES_REF, input_mode="File"),
        "zarr": TrainingInput(ZARR_REF, input_mode="FastFile")
    }
)

step_train_gat = TrainingStep(
    name="TrainGATModel",
    step_args=train_gat_args,
    cache_config=cache_config
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


# Pipeline Execution

In [ ]:
# CNN Training Job
pipeline = Pipeline(
    name="InteriorBoundsPipeline",
    steps=[step_train_cnn],
    sagemaker_session=pipeline_session,
)

' pipeline = Pipeline(\n    name="InteriorBoundsPipeline",\n    steps=[step_train_cnn],\n    sagemaker_session=pipeline_session,\n) '

In [ ]:
# GAT Training Job
pipeline = Pipeline(
    name="GATPipeline",
    steps=[step_train_gat],
    sagemaker_session=pipeline_session,
)

In [ ]:
pipeline.upsert(role_arn=role)
execution = pipeline.start()

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
